# Rebuild the ClawHub JIT dataset

This notebook rebuilds `clawhub_jit_dataset.parquet` from scratch:

1. Load the **frozen** Hugging Face snapshot (`OpenClaw/clawhub-security-signals-live`, pinned
   commit from `dataset_provenance.json`).
2. **Stage 1:** count versions per skill on ClawHub.
3. **Stage 2a:** fetch each candidate skill's version list and changelogs.
4. **Stage 2b:** fetch the security record of every version.
5. **Stage 3a/3b:** download version files and diff each version against the previous one.
6. Assemble everything into one parquet file.

Every crawl stage is **resumable**: results are appended to a checkpoint file as they arrive,
and re-running a cell skips anything already on disk. If the kernel restarts, just run the
notebook again from the top.

### What you need

- `dataset_provenance.json` from the original run, in the same folder. It holds the pinned
  Hugging Face commit. This notebook never resolves `main`.
- `pip install datasets huggingface_hub pandas pyarrow requests`

### Will the result be identical?

Close, but not byte for byte. The Hugging Face snapshot is pinned, so that part is exact.
The crawl, however, queries the **live** ClawHub API, which keeps changing:

- new versions get published (cut off below with `VERSION_CUTOFF_UTC`)
- the newest version of a skill gets re-scanned with ClawHub's current pipeline
- skills and versions can be deleted

So scanner results, especially on each skill's newest version, can differ from the original
run. For exact replication, use the published parquet. This notebook documents and repeats
how it was built.

### Rough run time

Stage 1 about 1 hour, Stage 2a about 30 minutes, Stage 2b several hours, Stage 3a several hours
at full scale. Times depend on your connection and ClawHub's rate limits.

## 0. Settings

In [1]:
import gzip
import io
import json
import re
import threading
import time
import zipfile
import difflib
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from IPython.display import display

pd.set_option("display.max_columns", None)

# ---- inputs / outputs
PROVENANCE_PATH = Path("dataset_provenance.json")      # from the original run (required)
COUNTS_PATH     = Path("clawhub_version_counts.jsonl")  # Stage 1 checkpoint
VERSIONS_PATH   = Path("clawhub_version_lists.jsonl")   # Stage 2a checkpoint
SECURITY_PATH   = Path("clawhub_version_security.jsonl")# Stage 2b checkpoint
CONTENT_DIR     = Path("clawhub_content")               # Stage 3a cache
CONTENT_INDEX   = Path("clawhub_content_index.jsonl")   # Stage 3a checkpoint
DIFFS_PARQUET   = Path("clawhub_diffs.parquet")
HISTORY_PARQUET = Path("clawhub_version_history.parquet")
OUTPUT_PARQUET  = Path("clawhub_jit_dataset.parquet")
OUTPUT_PREVIEW  = Path("clawhub_jit_dataset_preview.csv")
RUN_LOG_PATH    = Path("reproduction_provenance.json")  # written by this notebook

# Optional: the original parquet, to compare against at the end
REFERENCE_PARQUET = Path("reference/clawhub_jit_dataset.parquet")

# ---- crawl settings (same as the original run)
API = "https://clawhub.ai/api/v1"
MIN_VERSIONS = 2          # a skill needs 2+ versions to give at least one pair
MAX_RETRIES = 4
WORKERS_STAGE1 = 32
WORKERS_LATER = 48

# ---- Stage 3 (file downloads)
# The original dataset ran Stage 3a as a 50-skill pilot only. Keep 50 to match it,
# or set None for the full crawl (several hours, several GB of disk).
SMOKE_TEST_SKILLS = 50
MAX_FILE_BYTES = 1_000_000    # skip any single file larger than this
MAX_TOTAL_BYTES = 5_000_000   # cap per version

# ---- version lists longer than 50
# The versions endpoint returns at most 50 versions per request. Set True to run the
# optional paging cell after Stage 2a, which tries to fetch the rest. False matches the
# original dataset (latest 50 versions only for long-lived skills).
FETCH_BEYOND_50 = False
PAGED_PATH = Path("clawhub_version_lists_paged.jsonl")   # paging checkpoint
MAX_PAGES = 200                                          # safety stop per skill

## 1. Load the frozen snapshot

Loads by the pinned commit in `dataset_provenance.json`, never by `main`, so everyone gets the
same 77,873 rows.

`VERSION_CUTOFF_UTC` drops any version published after the original Stage 2a crawl finished.
Without it, versions published since then would be added and the pairs would change.

In [2]:
if not PROVENANCE_PATH.exists():
    raise FileNotFoundError(
        f"{PROVENANCE_PATH} is missing. Copy it from the original run: it holds the "
        f"pinned Hugging Face commit this notebook loads."
    )
provenance = json.loads(PROVENANCE_PATH.read_text())

from datasets import load_dataset

dataset = load_dataset(
    provenance["source_dataset"],
    name="default",
    revision=provenance["source_dataset_revision"],
    verification_mode="no_checks",   # the Hub's cached split sizes can be stale
)
df_all = dataset["latest"].to_pandas()

print(f"snapshot {provenance['source_dataset']} @ {provenance['source_dataset_revision'][:12]} "
      f"(frozen {provenance['source_dataset_frozen_at']})")
print(f"df_all: {len(df_all):,} rows x {len(df_all.columns)} columns")
assert len(df_all) == 77_873, f"expected 77,873 rows, got {len(df_all):,}"

# Cutoff: end of the original Stage 2a crawl, if recorded; otherwise the freeze time.
stage2a = (provenance.get("crawl") or {}).get("stage2a_version_lists") or {}
VERSION_CUTOFF_UTC = stage2a.get("last_written_utc") or provenance["source_dataset_frozen_at"]
CUTOFF_MS = int(pd.Timestamp(VERSION_CUTOFF_UTC).timestamp() * 1000)
print(f"versions published after {VERSION_CUTOFF_UTC} will be dropped")

snapshot OpenClaw/clawhub-security-signals-live @ 323d4b4bd899 (frozen 2026-09-25T09:01:35.555373+00:00)
df_all: 77,873 rows x 25 columns
versions published after 2026-09-25T06:54:44.257906+00:00 will be dropped


## 2. Shared crawl helpers

One HTTP session, one retry policy and one checkpoint format, used by every stage.

Retry policy: network errors and 5xx responses wait and retry, 429 (rate limited) waits for
`Retry-After`, 404 is recorded as `not_found`, anything else is recorded as `http_<code>`.
After `MAX_RETRIES` attempts the row is recorded as `failed_after_retries`.

In [3]:
_write_lock = threading.Lock()

session = requests.Session()
session.mount("https://", HTTPAdapter(pool_connections=WORKERS_LATER,
                                      pool_maxsize=WORKERS_LATER * 2))

def get_with_retries(url, params, timeout=15):
    # Returns (response or None, status string).
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(url, params=params, timeout=timeout)
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
            continue
        if resp.status_code == 200:
            return resp, "ok"
        if resp.status_code == 429:
            time.sleep(float(resp.headers.get("Retry-After", 2 * (attempt + 1))))
            continue
        if resp.status_code == 404:
            return None, "not_found"
        if 500 <= resp.status_code < 600:
            time.sleep(1.5 * (attempt + 1))
            continue
        return None, f"http_{resp.status_code}"
    return None, "failed_after_retries"

def load_keys(path, key_fn):
    # Keys of every row already in a checkpoint file.
    done = set()
    if path.exists():
        with path.open(encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    done.add(key_fn(json.loads(line)))
    return done

def append_row(path, row):
    with _write_lock:
        with path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")

def run_parallel(fn, jobs, workers, label, every):
    # Runs fn(*job) for every job in a thread pool, printing progress and ETA.
    processed, start = 0, time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(fn, *job) for job in jobs]
        for _ in as_completed(futures):
            processed += 1
            if processed % every == 0 or processed == len(jobs):
                rate = processed / max(time.time() - start, 1e-9)
                eta = (len(jobs) - processed) / rate / 60 if rate else float("inf")
                print(f"  {label}: {processed:,}/{len(jobs):,} ({rate:.1f}/s, ETA {eta:.1f} min)")
    print(f"{label} done.")

def split_slug(skill_slug):
    owner, _, slug = skill_slug.partition("/")
    return owner, slug

## 3. Stage 1: version count per skill

One request per skill in the snapshot. Only the total number of versions is kept, which is
enough to pick the skills worth crawling in depth.

In [4]:
def stage1_one(skill_slug):
    owner, slug = split_slug(skill_slug)
    resp, status = get_with_retries(f"{API}/skills/{slug}", {"ownerHandle": owner})
    versions = None
    if resp is not None:
        versions = ((resp.json().get("skill") or {}).get("stats") or {}).get("versions")
    append_row(COUNTS_PATH, {"skill_slug": skill_slug, "status": status, "versions": versions})

done = load_keys(COUNTS_PATH, lambda r: r["skill_slug"])
all_slugs = sorted(df_all["skill_slug"].unique())
todo = [(s,) for s in all_slugs if s not in done]
print(f"Stage 1: {len(done):,} already done, {len(todo):,} to go, {len(all_slugs):,} total")
run_parallel(stage1_one, todo, WORKERS_STAGE1, "Stage 1", every=500)

Stage 1: 77,873 already done, 0 to go, 77,873 total
Stage 1 done.


In [5]:
counts_df = pd.read_json(COUNTS_PATH, lines=True, convert_dates=False)
print(counts_df["status"].value_counts().to_string(), "\n")

candidate_slugs = sorted(
    counts_df.loc[(counts_df["status"] == "ok") & (counts_df["versions"] >= MIN_VERSIONS),
                  "skill_slug"].unique()
)
print(f"{len(candidate_slugs):,} skills have {MIN_VERSIONS}+ versions and go to Stage 2a")

status
ok                      77098
not_found                 639
failed_after_retries       86
http_400                   42
http_401                    8 

33,527 skills have 2+ versions and go to Stage 2a


## 4. Stage 2a: version lists and changelogs

One request per candidate skill. Returns each version's number, publish time and changelog.

Note: the API returns **at most 50 versions** per request, whatever `limit` is set to. Skills
with more than 50 versions therefore only get their 50 most recent. The original dataset has
this limitation. Set `FETCH_BEYOND_50 = True` in section 0 to run the optional paging cell
below, which tries to fetch the rest.

In [6]:
def stage2a_one(skill_slug):
    owner, slug = split_slug(skill_slug)
    resp, status = get_with_retries(f"{API}/skills/{slug}/versions",
                                    {"ownerHandle": owner, "limit": 200})
    items = (resp.json().get("items") or []) if resp is not None else []
    append_row(VERSIONS_PATH, {"skill_slug": skill_slug, "status": status,
                               "items": items, "hit_limit": len(items) >= 200})

done = load_keys(VERSIONS_PATH, lambda r: r["skill_slug"])
todo = [(s,) for s in candidate_slugs if s not in done]
print(f"Stage 2a: {len(done):,} already done, {len(todo):,} to go")
run_parallel(stage2a_one, todo, WORKERS_LATER, "Stage 2a", every=250)

Stage 2a: 33,527 already done, 0 to go
Stage 2a done.


### Optional: fetch versions beyond the first 50

Only runs when `FETCH_BEYOND_50 = True`.

The paging method is not documented. A first attempt that only followed `nextCursor`
stalled at 50, 100 or 150 versions for most skills, so this cell tries several methods per
skill and stops as soon as it has as many versions as Stage 1 counted:

1. `cursor`: follow `nextCursor` until it stops or repeats
2. `offset`: `offset=50, 100, ...`
3. `page`: `page=2, 3, ...`
4. `before`: ask for versions published before the oldest one we have

A method that adds nothing on its first try is abandoned straight away, so wrong guesses are
cheap. Whichever method worked last is tried first on the next skill. Versions are
de-duplicated by version number, so overlapping pages are harmless.

Results go to their own checkpoint (`clawhub_version_lists_paged.jsonl`). The next cell uses a
paged list in place of the original one whenever it has more versions. The table at the end
shows which method worked and how many skills are still incomplete.

In [7]:
def fetch_page(owner, slug, extra):
    resp, status = get_with_retries(f"{API}/skills/{slug}/versions",
                                    {"ownerHandle": owner, "limit": 50, **extra}, timeout=30)
    if resp is None:
        return None, None, status
    data = resp.json()
    return data.get("items") or [], data.get("nextCursor"), "ok"

def add_items(items, seen):
    new = 0
    for it in items:
        v = it.get("version")
        if v is not None and v not in seen:
            seen[v] = it
            new += 1
    return new

def by_cursor(owner, slug, seen, target):
    cursor, used = None, set()
    for _ in range(MAX_PAGES):
        items, nxt, st = fetch_page(owner, slug, {"cursor": cursor} if cursor else {})
        if st != "ok":
            return st
        new = add_items(items, seen)
        if len(seen) >= target or not nxt or nxt in used or (cursor and new == 0):
            return "ok"
        used.add(nxt)
        cursor = nxt
    return "ok"

def by_number(key, start, step):
    # offset-style (offset=50,100,...) or page-style (page=2,3,...) paging
    def run(owner, slug, seen, target):
        for k in range(MAX_PAGES):
            items, _, st = fetch_page(owner, slug, {key: start + k * step})
            if st != "ok":
                return st
            if add_items(items, seen) == 0 or len(seen) >= target:
                return "ok"
        return "ok"
    return run

def by_before(owner, slug, seen, target):
    for _ in range(MAX_PAGES):
        oldest = min(it.get("createdAt") or 0 for it in seen.values())
        items, _, st = fetch_page(owner, slug, {"before": oldest})
        if st != "ok":
            return st
        if add_items(items, seen) == 0 or len(seen) >= target:
            return "ok"
    return "ok"

STRATEGIES = {
    "cursor": by_cursor,
    "offset": by_number("offset", 50, 50),
    "page": by_number("page", 2, 1),
    "before": by_before,
}
order = list(STRATEGIES)   # reordered so the last method that worked goes first

def page_one(skill_slug, first_items, target):
    owner, slug = split_slug(skill_slug)
    seen = {}
    add_items(first_items, seen)
    worked, status = None, "ok"
    for name in list(order):
        before_n = len(seen)
        status = STRATEGIES[name](owner, slug, seen, target)
        if len(seen) > before_n:
            worked = name
            order.remove(name)
            order.insert(0, name)
        if len(seen) >= target:
            break
    return {"skill_slug": skill_slug, "status": status, "items": list(seen.values()),
            "expected": int(target), "got": len(seen), "method": worked,
            "complete": len(seen) >= target}

if FETCH_BEYOND_50:
    lists = pd.read_json(VERSIONS_PATH, lines=True, convert_dates=False)
    lists = lists[lists["status"] == "ok"]
    expected = counts_df.set_index("skill_slug")["versions"]
    lists = lists.assign(n=lists["items"].map(len),
                         expected=lists["skill_slug"].map(expected))
    truncated = lists[lists["expected"] > lists["n"]]

    done = load_keys(PAGED_PATH, lambda r: r["skill_slug"])
    todo = truncated[~truncated["skill_slug"].isin(done)]
    print(f"{len(truncated):,} skills have more versions than Stage 2a returned; "
          f"{len(done):,} already paged, {len(todo):,} to go\n")

    start = time.time()
    for i, r in enumerate(todo.itertuples(), 1):
        row = page_one(r.skill_slug, r.items, r.expected)
        append_row(PAGED_PATH, row)
        if i % 10 == 0 or i == len(todo):
            rate = i / max(time.time() - start, 1e-9)
            print(f"  {i:,}/{len(todo):,} ({rate:.2f} skills/s)  last: {r.skill_slug} "
                  f"{row['got']}/{row['expected']} via {row['method']}")

    if PAGED_PATH.exists():
        paged = pd.read_json(PAGED_PATH, lines=True, convert_dates=False)
        print("\nwhich method worked:")
        display(paged["method"].fillna("none worked").value_counts().rename("skills").to_frame())
        print(f"complete: {int(paged['complete'].sum()):,} of {len(paged):,} skills; "
              f"versions recovered: {int((paged['got'] - 50).clip(lower=0).sum()):,}")
else:
    print("FETCH_BEYOND_50 is False: skipping (skills with 50+ versions keep their latest 50)")

FETCH_BEYOND_50 is False: skipping (skills with 50+ versions keep their latest 50)


In [8]:
version_lists_df = pd.read_json(VERSIONS_PATH, lines=True, convert_dates=False)
print(version_lists_df["status"].value_counts().to_string(), "\n")

# Swap in paged lists where they hold more versions
if FETCH_BEYOND_50 and PAGED_PATH.exists():
    paged = (pd.read_json(PAGED_PATH, lines=True, convert_dates=False)
             .drop_duplicates("skill_slug", keep="last").set_index("skill_slug")["items"])
    swapped = 0
    for idx, r in version_lists_df.iterrows():
        if r["skill_slug"] in paged.index and len(paged[r["skill_slug"]]) > len(r["items"] or []):
            version_lists_df.at[idx, "items"] = paged[r["skill_slug"]]
            swapped += 1
    print(f"using paged version lists for {swapped:,} skill(s)\n")

rows = []
for r in version_lists_df.itertuples():
    if r.status != "ok":
        continue
    owner, slug = split_slug(r.skill_slug)
    for item in (r.items or []):
        rows.append({
            "skill_slug": r.skill_slug,
            "owner_handle": owner,
            "slug": slug,
            "version": item.get("version"),
            "created_at": item.get("createdAt"),          # epoch milliseconds
            "changelog_text": item.get("changelog"),
            "changelog_source": item.get("changelogSource"),
        })
versions_flat_df = pd.DataFrame(rows)

# Report the 50-per-request cap (checked before the cutoff, which also removes versions)
per_skill = versions_flat_df.groupby("skill_slug").size()
expected = counts_df.set_index("skill_slug")["versions"]
capped = per_skill[expected.reindex(per_skill.index) > per_skill]
print(f"{len(capped):,} skill(s) have fewer versions than Stage 1 counted "
      f"(the 50-per-request cap)")

# Drop versions published after the original crawl
late = versions_flat_df["created_at"] > CUTOFF_MS
print(f"dropping {int(late.sum()):,} version(s) published after {VERSION_CUTOFF_UTC}")
versions_flat_df = (versions_flat_df[~late]
                    .sort_values(["skill_slug", "created_at"])
                    .reset_index(drop=True))

print(f"\n{len(versions_flat_df):,} (skill, version) rows across "
      f"{versions_flat_df['skill_slug'].nunique():,} skills")

status
ok                      33492
failed_after_retries       32
not_found                   3 

613 skill(s) have fewer versions than Stage 1 counted (the 50-per-request cap)
dropping 0 version(s) published after 2026-09-25T06:54:44.257906+00:00

177,339 (skill, version) rows across 33,492 skills


## 5. Stage 2b: security record per version

One request per version. Keeps three scanners:

- **LLM scanner (ClawScan):** on almost every version. Verdict, confidence, five dimension
  ratings (ok / note / concern), and prose findings.
- **SkillSpector:** only on versions scanned after ClawHub added it (around May to June 2026).
- **VirusTotal:** meaning depends on `vt_source` ("palm" = model-based with prose,
  "engines" = antivirus counts).

Nested objects are stored as JSON strings so nothing is lost.

In [9]:
LLM_DIMENSIONS = [
    "purpose_capability",
    "instruction_scope",
    "install_mechanism",
    "environment_proportionality",
    "persistence_privilege",
]

def as_json(value):
    return json.dumps(value, ensure_ascii=False) if value else None

def parse_security(data):
    v = data.get("version") or {}
    sec = v.get("security") or {}
    scanners = sec.get("scanners") or {}
    ss = scanners.get("skillspector") or {}
    llm = scanners.get("llm") or {}
    vt = scanners.get("vt") or {}
    dims = {e["name"]: e.get("rating") for e in (llm.get("dimensions") or []) if e.get("name")}

    out = {
        "crawled_detail_status": "ok",
        # registry-level verdict
        "crawled_security_status": sec.get("status"),
        "crawled_security_checked_at": sec.get("checkedAt"),
        "crawled_has_scan_result": sec.get("hasScanResult"),
        "crawled_has_warnings": sec.get("hasWarnings"),
        "crawled_security_model": sec.get("model"),
        "crawled_sha256": sec.get("sha256hash"),
        # SkillSpector
        "crawled_skillspector_status": ss.get("status"),
        "crawled_skillspector_normalized_status": ss.get("normalizedStatus"),
        "crawled_skillspector_score": ss.get("score"),
        "crawled_skillspector_severity": ss.get("severity"),
        "crawled_skillspector_recommendation": ss.get("recommendation"),
        "crawled_skillspector_issue_count": ss.get("issueCount"),
        "crawled_skillspector_checked_at": ss.get("checkedAt"),
        # LLM scanner
        "llm_status": llm.get("status"),
        "llm_verdict": llm.get("verdict"),
        "llm_normalized_status": llm.get("normalizedStatus"),
        "llm_confidence": llm.get("confidence"),
        "llm_summary": llm.get("summary"),
        "llm_guidance": llm.get("guidance"),
        "llm_findings": llm.get("findings"),
        "llm_model": llm.get("model"),
        "llm_checked_at": llm.get("checkedAt"),
        "llm_dimensions_json": as_json(llm.get("dimensions")),
        "llm_risk_summary_json": as_json(llm.get("riskSummary")),
        "llm_agentic_risk_findings_json": as_json(llm.get("agenticRiskFindings")),
        # VirusTotal
        "vt_status": vt.get("status"),
        "vt_verdict": vt.get("verdict"),
        "vt_normalized_status": vt.get("normalizedStatus"),
        "vt_source": vt.get("source"),
        "vt_analysis": vt.get("analysis"),
        "vt_checked_at": vt.get("checkedAt"),
    }
    for name in LLM_DIMENSIONS:
        out[f"llm_dim_{name}"] = dims.get(name)
    return out

def stage2b_one(skill_slug, owner, slug, version):
    resp, status = get_with_retries(f"{API}/skills/{slug}/versions/{version}",
                                    {"ownerHandle": owner})
    result = parse_security(resp.json()) if resp is not None else {"crawled_detail_status": status}
    append_row(SECURITY_PATH, {"skill_slug": skill_slug, "version": version, **result})

done = load_keys(SECURITY_PATH, lambda r: (r["skill_slug"], r["version"]))
todo = [(r.skill_slug, r.owner_handle, r.slug, r.version)
        for r in versions_flat_df.itertuples() if (r.skill_slug, r.version) not in done]
print(f"Stage 2b: {len(done):,} already done, {len(todo):,} to go")
run_parallel(stage2b_one, todo, WORKERS_LATER, "Stage 2b", every=1000)

Stage 2b: 177,339 already done, 0 to go
Stage 2b done.


In [ ]:
details_df = pd.read_json(SECURITY_PATH, lines=True, convert_dates=False)
# If a version was retried, keep its last record
details_df = details_df.drop_duplicates(["skill_slug", "version"], keep="last")
print(details_df["crawled_detail_status"].value_counts().to_string(), "\n")

version_history_df = (versions_flat_df
                      .merge(details_df, on=["skill_slug", "version"], how="left")
                      .sort_values(["skill_slug", "created_at"])
                      .reset_index(drop=True))
version_history_df.to_parquet(HISTORY_PARQUET, index=False)
print(f"version history: {len(version_history_df):,} rows, "
      f"{version_history_df['skill_slug'].nunique():,} skills")

## 6. Stage 3a: download each version's files

One zip per version from `/api/v1/download`. Only UTF-8 text files are kept, with size caps.
Each version is cached as gzipped JSON so Stage 3b needs no network.

`SMOKE_TEST_SKILLS` (set in section 0) limits this to the first N skills alphabetically.

In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9._-]", "_", str(value))

def content_path(skill_slug, version):
    owner, slug = split_slug(skill_slug)
    return CONTENT_DIR / safe_name(owner) / f"{safe_name(slug)}__{safe_name(version)}.json.gz"

def extract_text_files(zip_bytes):
    files, total, truncated = {}, 0, False
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for info in sorted(zf.infolist(), key=lambda i: i.filename):
            if info.is_dir() or info.file_size > MAX_FILE_BYTES:
                continue
            try:
                text = zf.read(info).decode("utf-8")
            except (UnicodeDecodeError, RuntimeError):
                continue
            if total + len(text) > MAX_TOTAL_BYTES:
                truncated = True
                break
            files[info.filename] = text
            total += len(text)
    return files, total, truncated

def stage3a_one(skill_slug, owner, slug, version):
    resp, status = get_with_retries(f"{API}/download",
                                    {"slug": slug, "ownerHandle": owner, "version": version},
                                    timeout=60)
    result = {"status": status}
    if resp is not None:
        try:
            files, total, truncated = extract_text_files(resp.content)
            path = content_path(skill_slug, version)
            path.parent.mkdir(parents=True, exist_ok=True)
            with gzip.open(path, "wt", encoding="utf-8") as f:
                json.dump(files, f)
            result = {"status": "ok", "n_files": len(files), "n_chars": total,
                      "truncated": truncated}
        except zipfile.BadZipFile:
            result = {"status": "bad_zip"}
    append_row(CONTENT_INDEX, {"skill_slug": skill_slug, "version": version, **result})

target = versions_flat_df
if SMOKE_TEST_SKILLS is not None:
    keep = sorted(versions_flat_df["skill_slug"].unique())[:SMOKE_TEST_SKILLS]
    target = versions_flat_df[versions_flat_df["skill_slug"].isin(keep)]
    print(f"pilot: {len(keep)} skills / {len(target):,} versions "
          f"(set SMOKE_TEST_SKILLS = None for the full crawl)")

done = load_keys(CONTENT_INDEX, lambda r: (r["skill_slug"], r["version"]))
todo = [(r.skill_slug, r.owner_handle, r.slug, r.version)
        for r in target.itertuples() if (r.skill_slug, r.version) not in done]
print(f"Stage 3a: {len(done):,} already done, {len(todo):,} to go")
run_parallel(stage3a_one, todo, WORKERS_LATER, "Stage 3a", every=250)

if CONTENT_INDEX.exists():
    print(pd.read_json(CONTENT_INDEX, lines=True, convert_dates=False)["status"]
          .value_counts().to_string())

## 7. Stage 3b: diffs

No network. Walks each skill's versions in publish order and diffs each one against the
version before it, across every text file. If a version has no cached files, the next version
gets no diff either, so nothing is diffed against the wrong ancestor.

Also stores each version's `SKILL.md` text (or the largest `.md` file if there is none).

In [ ]:
def read_content(skill_slug, version):
    path = content_path(skill_slug, version)
    if not path.exists():
        return None
    try:
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    except (OSError, json.JSONDecodeError):
        return None

def pick_skill_md(files):
    if not files:
        return None
    for name in files:
        if name.split("/")[-1].upper() == "SKILL.MD":
            return files[name]
    md_names = [n for n in files if n.lower().endswith(".md")]
    return files[max(md_names, key=lambda n: len(files[n]))] if md_names else None

def bundle_diff(prev_files, curr_files):
    prev_files, curr_files = prev_files or {}, curr_files or {}
    chunks, added, removed = [], 0, 0
    for name in sorted(set(prev_files) | set(curr_files)):
        before = (prev_files.get(name) or "").splitlines(keepends=True)
        after = (curr_files.get(name) or "").splitlines(keepends=True)
        if before == after:
            continue
        hunk = list(difflib.unified_diff(before, after, fromfile=f"a/{name}", tofile=f"b/{name}"))
        for line in hunk:
            if line.startswith(("+++", "---", "@@")):
                continue
            if line.startswith("+"):
                added += 1
            elif line.startswith("-"):
                removed += 1
        chunks.extend(hunk)
    return "".join(chunks), added, removed

diff_rows, n_missing = [], 0
for skill_slug, group in versions_flat_df.groupby("skill_slug", sort=False):
    prev_files = None
    for r in group.itertuples():
        curr_files = read_content(skill_slug, r.version)
        if curr_files is None:
            n_missing += 1
            prev_files = None
            continue
        if prev_files is None:
            diff_text, added, removed = None, None, None
        else:
            diff_text, added, removed = bundle_diff(prev_files, curr_files)
        diff_rows.append({
            "skill_slug": skill_slug, "version": r.version, "diff_text": diff_text,
            "lines_added": added, "lines_removed": removed,
            "lines_changed": None if added is None else added + removed,
            "crawled_skill_content": pick_skill_md(curr_files),
        })
        prev_files = curr_files

diffs_df = pd.DataFrame(diff_rows, columns=[
    "skill_slug", "version", "diff_text", "lines_added", "lines_removed",
    "lines_changed", "crawled_skill_content",
])
diffs_df.to_parquet(DIFFS_PARQUET, index=False)
print(f"{len(diffs_df):,} versions with files, {int(diffs_df['diff_text'].notna().sum()):,} "
      f"with a diff, {n_missing:,} without cached files")

## 8. Assemble the dataset

One row per (skill, version), ordered by publish time within each skill (not by semver).

Added columns: position in the skill's history, previous version, days since the previous
version, scan lag, and the previous version's security results (so pair deltas need no
re-crawl). The snapshot's own columns are attached only to the version the snapshot recorded;
other rows leave them empty.

In [ ]:
vh = version_history_df.sort_values(["skill_slug", "created_at"]).copy()
g = vh.groupby("skill_slug")

vh["version_index"] = g.cumcount() + 1
vh["prev_version"] = g["version"].shift(1)
vh["days_since_prev_version"] = (vh["created_at"] - g["created_at"].shift(1)) / 86_400_000.0
vh["llm_scan_lag_days"] = (vh["llm_checked_at"] - vh["created_at"]) / 86_400_000.0

CARRY = ["crawled_skillspector_score", "crawled_skillspector_severity",
         "llm_normalized_status", "llm_model"] + [f"llm_dim_{n}" for n in LLM_DIMENSIONS]
for col in CARRY:
    vh[f"prev_{col}"] = g[col].shift(1)

vh = vh.merge(diffs_df, on=["skill_slug", "version"], how="left")

final_df = vh.merge(df_all, left_on=["skill_slug", "version"],
                    right_on=["skill_slug", "skill_version"], how="left")

crawled_cols = (
    ["skill_slug", "owner_handle", "slug", "version", "version_index",
     "created_at", "prev_version", "days_since_prev_version",
     "changelog_text", "changelog_source",
     "diff_text", "lines_added", "lines_removed", "lines_changed",
     "crawled_skill_content", "crawled_sha256",
     "llm_status", "llm_verdict", "llm_normalized_status", "llm_confidence"]
    + [f"llm_dim_{n}" for n in LLM_DIMENSIONS]
    + ["llm_dimensions_json", "llm_risk_summary_json", "llm_agentic_risk_findings_json",
       "llm_summary", "llm_guidance", "llm_findings",
       "llm_model", "llm_checked_at", "llm_scan_lag_days",
       "crawled_skillspector_status", "crawled_skillspector_normalized_status",
       "crawled_skillspector_score", "crawled_skillspector_severity",
       "crawled_skillspector_recommendation", "crawled_skillspector_issue_count",
       "crawled_skillspector_checked_at",
       "vt_status", "vt_verdict", "vt_normalized_status", "vt_source",
       "vt_analysis", "vt_checked_at",
       "crawled_security_status", "crawled_security_checked_at",
       "crawled_security_model", "crawled_has_scan_result",
       "crawled_has_warnings", "crawled_detail_status"]
    + [f"prev_{c}" for c in CARRY]
)
hf_cols = [c for c in df_all.columns if c != "skill_slug"]
keep = [c for c in crawled_cols + hf_cols if c in final_df.columns]
dropped = [c for c in final_df.columns if c not in keep]
if dropped:
    print(f"WARNING: dropping {len(dropped)} column(s) not in the keep list: {dropped}")
final_df = final_df[keep]

final_df.to_parquet(OUTPUT_PARQUET, index=False)

bulky = {"diff_text", "crawled_skill_content", "llm_dimensions_json", "llm_risk_summary_json",
         "llm_agentic_risk_findings_json", "llm_summary", "llm_guidance", "llm_findings",
         "vt_analysis", "skill_md_content", "skill_bundle_content",
         "clawscan_context", "clawscan_summary"}
final_df[[c for c in final_df.columns if c not in bulky]].to_csv(OUTPUT_PREVIEW, index=False)

print(f"saved {OUTPUT_PARQUET}: {len(final_df):,} rows x {len(final_df.columns)} columns, "
      f"{final_df['skill_slug'].nunique():,} skills")
print(f"rows carrying the snapshot's columns: {int(final_df['skill_version'].notna().sum()):,}")

## 9. Record this run

Writes `reproduction_provenance.json`: the pinned snapshot, the cutoff, the settings, and when
each checkpoint was last written. The original `dataset_provenance.json` is left untouched.

In [ ]:
def describe(p):
    if not p.exists():
        return {"exists": False}
    with p.open(encoding="utf-8") as f:
        rows = sum(1 for line in f if line.strip())
    return {"exists": True, "rows": rows,
            "size_mb": round(p.stat().st_size / 1_048_576, 1),
            "last_written_utc": datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).isoformat()}

run_log = {
    "source_dataset": provenance["source_dataset"],
    "source_dataset_revision": provenance["source_dataset_revision"],
    "version_cutoff_utc": VERSION_CUTOFF_UTC,
    "settings": {"min_versions": MIN_VERSIONS, "smoke_test_skills": SMOKE_TEST_SKILLS,
                 "fetch_beyond_50": FETCH_BEYOND_50,
                 "max_file_bytes": MAX_FILE_BYTES, "max_total_bytes": MAX_TOTAL_BYTES},
    "crawl": {name: describe(p) for name, p in {
        "stage1_version_counts": COUNTS_PATH,
        "stage2a_version_lists": VERSIONS_PATH,
        "stage2b_security": SECURITY_PATH,
        "stage3a_content_index": CONTENT_INDEX,
    }.items()},
    "output": {"rows": len(final_df), "columns": len(final_df.columns),
               "skills": int(final_df["skill_slug"].nunique())},
    "recorded_utc": datetime.now(timezone.utc).isoformat(),
}
RUN_LOG_PATH.write_text(json.dumps(run_log, indent=2))
print(json.dumps(run_log, indent=2))

## 10. Optional: compare with the original parquet

If the original `clawhub_jit_dataset.parquet` is placed at `REFERENCE_PARQUET`, this shows how
close the rebuild is: which (skill, version) rows match, and how often the main labels agree on
the rows both have. Differences are expected on newest versions (re-scanned since).

In [ ]:
if REFERENCE_PARQUET.exists():
    KEY = ["skill_slug", "version"]
    COMPARE = ["llm_normalized_status", "crawled_skillspector_normalized_status",
               "vt_normalized_status", "changelog_text", "crawled_sha256"]
    ref = pd.read_parquet(REFERENCE_PARQUET, columns=KEY + COMPARE)
    new = final_df[KEY + COMPARE]

    both = ref.merge(new, on=KEY, how="outer", suffixes=("_ref", "_new"), indicator=True)
    print("row overlap (left_only = only in original, right_only = only in rebuild):")
    print(both["_merge"].value_counts().to_string(), "\n")

    matched = both[both["_merge"] == "both"]
    out = []
    for c in COMPARE:
        a, b = matched[f"{c}_ref"], matched[f"{c}_new"]
        same = (a == b) | (a.isna() & b.isna())
        out.append({"column": c, "rows_compared": len(matched), "agree": f"{same.mean():.2%}"})
    display(pd.DataFrame(out))
else:
    print(f"no reference parquet at {REFERENCE_PARQUET}, skipping")